In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# help locating the package sgpykit (comment this out if you installed it)
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

import sgpykit as sg

import logging
from sgpykit.util.log import logger

logging.basicConfig(
    # see https://docs.python.org/3/library/logging.html#logrecord-attributes
    format="%(levelname)s %(name)s:%(lineno)d: %(message)s",
)
# switching log level for sgpykit. Also see https://docs.python.org/3/library/logging.html#logging-levels
logger.setLevel(logging.INFO)
# logger.setLevel(logging.DEBUG)

# sgpykit

## Differences to Sparse Grids Matlab Kit

- sgpykit index arrays are in a 0-based index format (Matlab uses 1-based indexing), so the first element of an array has index 0.
- Scalars are not arrays. The default array type is float64, but you can specify the dtype and number of dimensions.
- Use placeholders (`_`) for unused return arguments like `something, _, another_thing = some_function()` (Matlab allows a flexible number of output arguments, while sgpykit/Python does not).
- Use named arguments in functions to specify variables, as in `evaluate_on_sparse_grid(f, S=None, Sr=my_Sr)`.
- More information about such differences can be found in [differences.md](../differences.md).

In [ ]:
N = 2
w = 3
idxset = lambda i: sum(i-1)
sg.multiidx_gen(N=N, rule=idxset, w=w, base=1)

In [ ]:
# for 0-based indices
idxset = lambda i: sum(i)
sg.multiidx_gen(N=N, rule=idxset, w=w, base=0)

# PART 1 - INTRODUCTION

## What Is a Sparse Grid?

A sparse grid is a linear combination of many tensor grids on $\mathbb{R}^N$ (parameter space).
Each of the tensor grids included has "few points." With suitable linear combinations
of such grids, it is possible to achieve good accuracy in quadrature and interpolation,
while reducing the computational cost compared to using a single tensor grid.

Run these commands to build a sparse grid and visualize each component.

In [ ]:
N = 2
w = 3

knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
S, C = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
S[0]
#len(S)
#S.shape
#S.knots


### Visualisation

- For marker styles, see the [matplotlib API](https://matplotlib.org/stable/api/markers_api.html).

In [ ]:
# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')
axs.set_title("Knots");

In [ ]:
# each component
fig, axs = sg.figure_create(nrows=2, ncols=4, figsize=(16, 8))
s_max = len(S)
k = 0
for s in range(s_max):
    if len(S[s])>0:
        sg.plot_sparse_grid(axs.flat[k], S[s], [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k');
        k = k + 1
fig.tight_layout()  # auto spacing subplots

### 3D Plot

In [ ]:
N = 3
w = 3
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
S3, C = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)

In [ ]:
#%matplotlib ipympl
# switch to ipympl matplotlib GUI backend for Jupyter, to be able to rotate 3D view (a bit laggy though)

sg.plot3_sparse_grid(S3, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

In [ ]:
%matplotlib inline

## Tensor Grid

In [ ]:
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
lev2knots = sg.lev2knots_doubling
#lev2knots = sg.lev2knots_lin
N = 3
ii = np.array([1, 2, 3])
m = sg.lev2knots_doubling(ii)
s = sg.tensor_grid(N, m, knots)
s

## Ingredients of a Sparse Grid

### 1D Knots

Each tensor grid in the sparse grid is built by taking Cartesian products of 1D point distributions.
In general, a different number of points is used in each direction. sgpykit provides
several knot families. These functions also return the quadrature weights associated with the knots
(more on this later).

**Gauss-Legendre points**: quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points.

In [ ]:
# switch back to inline matplotlib GUI backend for Jupyter
%matplotlib inline
sg.clear()  # clear previous plots in cache, otherwise they would appear again in this cell

n=5; a=1; b=4;
x, w = sg.knots_uniform(n, a, b)

fig, axs = sg.figure_create(figsize=(10,8))
sg.plot(axs, x, 0*x, 'ok','MarkerFaceColor','k','DisplayName','5 GL points')

**Clenshaw-Curtis points**: nested quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points. If one "doubles" the number of points, the new points will include the old ones.

In [ ]:
n=5; a=1; b=4;
x, w = sg.knots_CC(n, a, b)

#fig, axs = figure_create()
sg.plot(axs, x,1 + 0*x,'or','MarkerFaceColor','r','DisplayName','5 CC points')
fig

In [ ]:
fig, axs = sg.figure_create()
n=9; a=1; b=4;
x,_ = sg.knots_uniform(n,a,b)
sg.plot(axs, x,-1+0*x,'ob','MarkerFaceColor','b','DisplayName','9 GL points (does NOT includes the 5 points)')

n=9; a=1; b=4;
x,_ = sg.knots_CC(n,a,b)
sg.plot(axs, x, 2+0*x,'og','MarkerFaceColor','g','DisplayName','9 CC points (includes the 5 points)')

axs.set_ylim([-1.5, 4])
plt.legend()

**Leja points**: nested quadrature points to approximate integrals of the form
$$
\int_a^b f(x) \frac{1}{b-a} \, dx
$$
with $n$ points. Three different kinds of Leja points are available: Line Leja, sym-Line Leja, and p-disk Leja (see leja_points.m for more details). All Leja points are nested by construction.

In [ ]:
n=5; a=1; b=4;
x, _ = sg.knots_leja(n, a, b, 'line')

fig, axs = sg.figure_create()
sg.plot(axs, x, 1 + 0*x, 'or', 'MarkerFaceColor', 'r', 'DisplayName', '5 CC points')

n=9; a=1; b=4;
x, _ = sg.knots_leja(n, a, b, 'line')

sg.plot(axs, x, 2 + 0*x, 'og', 'MarkerFaceColor', 'g', 'DisplayName', '9 Line Leja points')  # TODO: changed to green
plt.legend()

In [ ]:
# ------- sym leja ----------
n=5; a=1; b=4;
x, _ = sg.knots_leja(n,a,b,'sym_line')

fig, axs = sg.figure_create()
sg.plot(axs, x, 3 + 0*x,'ok','MarkerFaceColor','k','DisplayName','5 sym-line Leja points')

n=9; a=1; b=4;
x, _ = sg.knots_leja(n,a,b,'sym_line')
sg.plot(axs, x, 4 + 0*x,'og','MarkerFaceColor','g','DisplayName','9 sym-Line Leja points')  # TODO: changed to green
plt.legend()

In [ ]:
# ------- p-disk leja ----------
n=5; a=1; b=4;
x,_ = sg.knots_leja(n,a,b,'p_disk')

fig, axs = sg.figure_create()
sg.plot(axs, x,5 + 0*x,'ob','MarkerFaceColor','b','DisplayName','5 p-Disk Leja points')

n=9; a=1; b=4;
x,_ = sg.knots_leja(n,a,b,'p_disk')
sg.plot(axs, x,6 + 0*x,'og','MarkerFaceColor','g','DisplayName','9 p-Disk Leja points')  # TODO: changed to green

axs.set_ylim([-1.5, 12])
plt.legend()

We also provide equispaced (trapezoidal quadrature rule) and midpoint points,
which should of course be used sparingly as they are not suitable for high-order
interpolation (Runge phenomenon).

In [ ]:
fig, axs = sg.figure_create()  # TODO: \/ changed point symbols

n=5; a=1; b=4;
x,_ = sg.knots_midpoint(n,a,b)
sg.plot(axs, x, 1 + 0*x,'or','MarkerFaceColor','r','DisplayName','5 midpoints')

n=6; a=1; b=4;
x,_ = sg.knots_trap(n,a,b)
sg.plot(axs, x,1 + 0*x,'ob','MarkerFaceColor','b','DisplayName','6 trapezoidal points')


n=10; a=1; b=4;
x,_ = sg.knots_midpoint(n,a,b)
sg.plot(axs, x,2 + 0*x,'xr','MarkerFaceColor','r','DisplayName','10 midpoints')

n=11; a=1; b=4;
x,_ = sg.knots_trap(n,a,b)
sg.plot(axs, x,2 + 0*x,'xb','MarkerFaceColor','b','DisplayName','11 trapezoidal points (nested with 6, 6+5=11)')

n=30; a=1; b=4;
x,_ = sg.knots_midpoint(n,a,b)
sg.plot(axs, x,3 + 0*x,'.r','MarkerFaceColor','r','DisplayName','30 midpoints (nested with 10 midpoints)')


axs.set_ylim([-0.5, 4.5])
plt.legend()

Quadrature points to approximate integrals with normal PDF, like
$$
\frac{1}{\sqrt{2 \sigma^2 \pi}} \int_\mathbb{R} f(x) \exp\left( -\frac{(x-\mu)^2}{2 \sigma^2} \right) \, dx
$$

In [ ]:
# Gauss-Hermite points
n=9; mu=0; sig=1;
x,_ = sg.knots_normal(n,mu,sig)

fig, axs = sg.figure_create()
sg.plot(axs, x, 0*x, 'ok','MarkerFaceColor','k','DisplayName','9 GH points')

# Genz-Keister / Kronrod - Patterson Nodes : nested quadrature points to approximate integrals as the previous

n=3;
x,_ = sg.knots_GK(n,0,1)
sg.plot(axs, x,1 + 0*x, 'or','MarkerFaceColor','r','DisplayName','3 GK points')

n=9;
x,_ = sg.knots_GK(n,0,1)
sg.plot(axs, x, 2 + 0*x, 'ob','MarkerFaceColor','b','DisplayName','9 GK points')

plt.legend()

In [ ]:
# Normal-Leja : nested quadrature points to approximate integrals as the previous
fig, axs = sg.figure_create()
mu=0; sig=1;
n=3;
x,_ = sg.knots_normal_leja(n,mu,sig,'line'); # another option here is 'sym_line'
sg.plot(axs, x,3 + 0*x,'xr','LineWidth',2,'MarkerFaceColor','r','DisplayName','3 Normal-Leja points')

n=9;
x,_ = sg.knots_normal_leja(n, mu,sig,'line');
sg.plot(axs, x, 4 + 0*x,'xb','LineWidth',2,'MarkerFaceColor','b','DisplayName','9 Normal-Leja points')

axs.set_ylim([-1.5, 7])
plt.legend()

Quadrature points to approximate integrals with exponential PDF, like
$$
\int_0^{\infty} f(x) \lambda e^{ -\lambda x } \, dx
$$

In [ ]:
# Gauss-Laguerre points
n=12;
lmbda = 1;
x,_ = sg.knots_exponential(n,lmbda);

fig, axs = sg.figure_create()
sg.plot(axs, x, 0*x, 'ok','MarkerFaceColor','k','DisplayName','12 Gauss-Laguerre points')

# and their Leja Counter part

x,_ = sg.knots_exponential_leja(n,lmbda);
sg.plot(axs, x, 1 + 0*x, 'or','LineWidth',2,'MarkerFaceColor','r','DisplayName','12 Exponential-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 1.5])
plt.legend()

Quadrature points to approximate integrals with gamma PDF, like
$$
\int_0^\infty f(x) \frac{\beta^{\alpha+1}}{\Gamma(\alpha+1)} \cdot x^\alpha \cdot e^{-\beta x} \, dx
$$

In [ ]:
#  Gauss-Generalized Laguerre points:  with n points
n=12;
alpha = 1; beta =2;

x,_ = sg.knots_gamma(n,alpha,beta);

fig, axs = sg.figure_create()
sg.plot(axs, x, 0*x, 'ok','MarkerFaceColor','k','DisplayName','12 Generalized Gauss-Laguerre points')

# and their Leja Counter part

x,_ = sg.knots_gamma_leja(n,alpha,beta);
sg.plot(axs, x,1 + 0*x,'or','LineWidth',2,'MarkerFaceColor','r','DisplayName','12 Gamma-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 1.5])
plt.legend()

Quadrature points to approximate integrals with beta PDF, like

$$
\int_{x_a}^{x_b} f(x) \frac{\Gamma(\alpha+\beta+2)}{\Gamma(\alpha+1) \cdot \Gamma(\beta+1) \cdot (x_b-x_a)^{\alpha+\beta+1}} \cdot (x-x_a)^\alpha \cdot (x_b-x)^\beta \, dx
$$


In [ ]:
#  Gauss-Jacobi points: with n points
n=12;
x_a = 1; x_b = 3;
alpha = -0.5; beta = 0.5;

x,_ = sg.knots_beta(n,alpha,beta,x_a,x_b);

fig, axs = sg.figure_create()
sg.plot(axs, x,0*x,'ok','MarkerFaceColor','k','DisplayName','12 Gauss-Jacobi points')

# and their Leja Counter part

n=12;
x,_ = sg.knots_beta_leja(n,alpha,beta,x_a,x_b,'line');
sg.plot(axs, x,1 + 0*x,'or','LineWidth',2,'MarkerFaceColor','r','DisplayName','12 Beta-Leja points')

n=12;
x,_ = sg.knots_beta_leja(n,alpha,beta,x_a,x_b,'sym_line'); # recommended only if alpha = beta
sg.plot(axs, x,2 + 0*x,'ob','LineWidth',2,'MarkerFaceColor','b','DisplayName','12 Sym. Beta-Leja points')

axs.grid(True)
axs.set_ylim([-0.5, 3])
plt.legend()

**Triangular-Leja points**: quadrature points to approximate integrals with triangular PDF (i.e., a linear decreasing PDF over the interval $[a,b]$), like
$$
\int_a^b f(x) \cdot \frac{2}{(b-a)^2} \cdot (b-x) \, dx
$$

In [ ]:
n = 12;
a = 0; b = 2;
x,_ = sg.knots_triangular_leja(n,a,b);

fig, axs = sg.figure_create()
sg.plot(axs, x, 0*x, 'ok','MarkerFaceColor','k','DisplayName','12 Triangular-Leja points')

axs.grid(True)
plt.legend()

### lev2knots Function

In view of building sparse grids, it is useful to order quadrature/interpolation rules in sequences, i.e. to
introduce levels for the rules. sgpykit provides 5 functions to this end:

### lev2knots_lin

Adds 1 point from one level to the next: consecutive quadrature/interpolation rules have
1, 2, 3, 4, ... points.

In [ ]:
sg.lev2knots_lin([1, 2, 3, 4, 5])


In [ ]:
# -> lev2knots_2step
#
# adds 2 points from one level to the next: consecutive quadrature/interpolation rules have
# 1,3,5,7,..points

sg.lev2knots_2step([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_doubling
#
# "doubles" the number of points from one level to the next: consecutive rules have 1,3,5,9,17... points

sg.lev2knots_doubling([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_tripling
#
# triples the number of points from one level to the next: consecutive rules have 1,3,9,27... points.


sg.lev2knots_tripling([1, 2, 3, 4, 5])

In [ ]:
# -> lev2knots_GK
#
# needed when using GK knots which are tabulated. consecutive rules have 1,3,9,19,35 points. The latter
# is the finest resolution possible

sg.lev2knots_GK([1, 2, 3, 4, 5])

### Multi-Index Set

The last ingredient to specify when building a sparse grid is the set of tensor grids to be used. The
algorithm will then take care of computing the coefficients of the linear combination of these grids
(note that such coefficients may be 0 as well).

The most convenient way to specify tensor grids is to use multi-index notation: every grid is
associated with a multi-index, that is a vector of integer numbers.
Each number in the vector tells the level of the quadrature rule used in each direction
of the parameter space. For example, the multi-index $[3, 5]$ is associated with the tensor grid
using a quadrature rule of level 3 in direction 1 and level 5 in direction 2. The actual number of points in
each direction depends on the level-knots relation specified by the lev2knots_\*\*\* function.

In [ ]:
N=2;
ii=[3, 5];
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')

S_lin = sg.tensor_grid(N, sg.lev2knots_lin(ii), knots);
S_doub = sg.tensor_grid(N, sg.lev2knots_doubling(ii), knots);

fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_doub, [],'color','r','marker','s','MarkerFaceColor','r','DisplayName','lev2knots-nested');
sg.plot_sparse_grid(axs, S_lin,[],'color','k','marker','o','MarkerFaceColor','k','DisplayName','lev2knots-lin');
plt.legend()

There are two ways of specifying the set of multi-indices to be used.

1) The first one is to use the parameters "level" and "idxset" of the function [`create_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid). In this case, the multi-index set will include all the multi-indices that satisfy the inequality

$$
\text{idxset}(\mathbf{i}) \leq \text{level}
$$

By default, idxset is set to $\text{idxset}(\mathbf{i}) = \sum (i_n - 1)$. The combination of idxset function and lev2knots function
defines the sparse grid type: using $\text{idxset}(\mathbf{i}) = \sum (i_n - 1)$ with lev2knots_lin results in the so-called TD
(Total Degree) tensor grid, while using it with lev2knots_doubling gives the original SM (Smolyak) grid.
Some choices are available by using the function

$$
[\text{lev2nodes}, \text{idxset}] = \text{define\_functions\_for\_rule}(\text{rule}, \text{rates})
$$

but any other set satisfying the so-called "admissibility condition" (see e.g. Gerstner-Griebel "Dimension-Adaptive Tensor-Product Quadrature") can be used.

In [ ]:
N = 2;
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')
w = 5; # level

lev2knots, idxset = sg.define_functions_for_rule('TD', N);
S_TD, _ = sg.create_sparse_grid(N,w,knots,lev2knots,idxset); # grid

lev2knots, idxset = sg.define_functions_for_rule('HC', N);
S_HC, _ = sg.create_sparse_grid(N,w,knots,lev2knots,idxset); # grid

# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_TD,[],'color','k','marker','o','MarkerFaceColor','k','DisplayName','TD-grid');
plt.legend()

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_HC,[],'color','k','marker','o','MarkerFaceColor','k','DisplayName','HC-grid');
plt.legend()

2) The second one is to use the function [`create_sparse_grid_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid_multiidx_set), where one specifies exactly
the set of multi-indices that one wishes to use. Again, the set has to satisfy
the "admissibility condition," and the rows have to be in lexicographic order.

In [ ]:
np.all(C[:-1] <= C[1:], axis=1)

In [ ]:
C=[
    [0, 0],
    [0, 1],
    [0, 2],
    [0, 3],
    [1, 0],
    [1, 1],
]
adm, C = sg.check_set_admissibility(C)

N=2
knots = lambda n: sg.knots_uniform(n, -1, 1, 'nonprob')
lev2knots, idxset = sg.define_functions_for_rule('HC', N)

S_M, _ = sg.create_sparse_grid_multiidx_set(C, knots, lev2knots)

fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S_M,[],'color','b','marker','d','MarkerFaceColor','b', 'DisplayName', 'Multiidx set')
axs.set_xlim([-1, 1])
axs.set_ylim([-1, 1])
plt.legend()

The package provides two functions to generate multi-index sets.

a) [`multiidx_box_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.multiidx_box_set) generates all multi-indices $\mathbf{j}$ that are component-wise less than or
equal to some other index $\mathbf{i}$. The minimal value of the components of the indices to be generated can be either 0 or 1. For instance:

In [ ]:
jj=[2, 3]
C,_ = sg.multiidx_box_set(jj, 0)
D,_ = sg.multiidx_box_set(jj, 1)

# the package comes with a convenience function to plot a multiidx set

fig, axs = sg.figure_create()
sg.plot_multiidx_set(axs, C,'xr','MarkerFaceColor','r','LineWidth',2,'MarkerSize',12,'DisplayName','Multiidx box set, min=0')
sg.plot_multiidx_set(axs, D,'ok','MarkerFaceColor','k','DisplayName','Multiidx box set, min=1')

axs.set_xlim([-0.5, 4])
axs.set_ylim([-0.5, 4])
plt.legend()

b) [`multiidx_gen()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.multiidx_gen) generates the set of all indices $\mathbf{i}$ such that $\text{rule}(\mathbf{i}) \leq w$, where rule is a function that takes as input a row vector
 (or a matrix where each multi-index is stored as a row) and returns a scalar value (or a column vector with the result of the operation applied
 to each row of the input index vector). Again, the minimum index can be 0 or 1.

In [ ]:
N=2
w=7
rule = lambda I: sum(I)
E = sg.multiidx_gen(N, rule, w, 0)
F = sg.multiidx_gen(N, rule, w, 1)

fig, axs = sg.figure_create()
sg.plot_multiidx_set(axs, E,'xr','MarkerFaceColor','r','LineWidth',2,'MarkerSize',12,'DisplayName','Multiidx gen, min=0')
sg.plot_multiidx_set(axs, F,'ok','MarkerFaceColor','k','DisplayName','Multiidx gen, min=1')

axs.set_xlim([-0.5, 8])
axs.set_ylim([-0.5, 8])
plt.legend()

Incidentally, [`plot_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.plot_multiidx_set) works also for $N=3$. For larger dimensions, one needs to input the subset of dimensions that are to be plotted.

In [ ]:
G,_ = sg.multiidx_box_set([2, 3, 5], 1);
fig, axs = sg.figure_create(dims=3)
sg.plot_multiidx_set(axs, G)

In [ ]:
H,_ = sg.multiidx_box_set([2, 3, 5, 4], 1);
fig, axs = sg.figure_create(dims=3)
sg.plot_multiidx_set(axs, H[:,[1, 3]])

When building a large sparse grid, it might be useful to recycle from previous grids to speed up the computation.

In [ ]:
knots = lambda n: sg.knots_normal(n,0,1)
lev2knots = sg.lev2knots_lin

N=40
w=4
S, _ = sg.create_sparse_grid(N,w,knots,lev2knots,lambda i: np.prod(i+1))

In [ ]:
logger.setLevel(logging.DEBUG)

w = 5

with sg.misc.time_block('build grid without recycling'):
    T,_ = sg.create_sparse_grid(N,w,knots,lev2knots,lambda i: np.prod(i+1))

with sg.misc.time_block('build grid with recycling'):
    T_rec,_ = sg.create_sparse_grid(N,w,knots,lev2knots,lambda i: np.prod(i+1), S)

assert T.isequal_to(T_rec), "Results Mismatch"

This is useful in iterative loops like:

In [ ]:
N=20
w=4
knots = lambda n: sg.knots_normal(n,0,1)
lev2knots = sg.lev2knots_lin

logger.setLevel(logging.INFO)

with sg.misc.time_block():
    for w in range(7):
        # build grid
        T,_ = sg.create_sparse_grid(N,w,knots,lev2knots,lambda i: np.prod(i+1))
        # then do something ...


with sg.misc.time_block():
    T_old = []
    for w in range(7):
        # build grid
        T,_ = sg.create_sparse_grid(N,w,knots,lev2knots,lambda i: np.prod(i+1),T_old)
        T_old = T
        # then do something ...


The same functionality is also available for [`create_sparse_grid_multiidx_set()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.create_sparse_grid_multiidx_set).

In [ ]:
N=5
w=4
knots = lambda n: sg.knots_normal(n,0,1)
lev2knots = sg.lev2knots_lin

ibox = [3, 4, 2, 4, 2]
_,C = sg.multiidx_box_set(ibox, 0)
D,_ = sg.matlab.sortrows(np.vstack((C, [1, 4, 1, 1, 5])))

S,_ = sg.create_sparse_grid_multiidx_set(C,knots,lev2knots)

In [ ]:
logger.setLevel(logging.DEBUG)

with sg.misc.time_block():
    T,_ = sg.create_sparse_grid_multiidx_set(D,knots,lev2knots)

with sg.misc.time_block():
    T_rec,_ = sg.create_sparse_grid_multiidx_set(D,knots,lev2knots,S)

assert T.isequal_to(T_rec), "Results Mismatch"

## Data Structures

A sparse grid is represented as a vector of structures. Each element is a tensor grid, with fields
containing the knots, the corresponding integration weights, its coefficient in the linear combination,
and the number of points.

Each tensor grid structure contains the following fields (also see the [SGMK manual online](https://sites.google.com/view/sparse-grids-kit#h.p_IBOqq8I5k39K)):

- **idx**: the 0-based multi-index $\mathbf{i} \in \mathcal{I}\subset\mathbb{N}^N$ corresponding to the current tensor grid
- **knots**: matrix collecting the knots $\mathcal{T}_{\mathbf{i}}$, each knot being a row vector
- **weights**: vector of the quadrature weights $\omega_{m(\mathbf{i})}^{(\mathbf{j})}$ corresponding to the knots
- **size**: size of the tensor grid, i.e. the number of knots $M_{\mathbf{i}} = \prod_{n=0}^{N-1} m(i_n)$
- **knots_per_dim**: cell array with $N$ components, each component collecting in an array the set of one-dimensional knots $\mathcal{T}_{n, i_n}$ used to build the tensor grid
- **m**: vector collecting the number of knots used in each of the $N$ directions $m(\mathbf{i}) = [m(i_0), m(i_1), \ldots, m(i_{N-1})]$
- **coeff**: the coefficients $c_{\mathbf{i}}$ of the sparse grid in the combination technique formulas

Note that sparse grid knots (and in general knots $\mathbf{y} \in \Gamma$) are always stored in sgpykit as row vectors 
(and sets of knots such as $\mathbf{S}.\text{knots}$ are stored as matrices where knots are rows).

Reduced sparse grids contain the following fields:

- **knots**: matrix collecting the list of non-repeated knots, i.e., the set $\mathcal{T}_{\mathcal{I}}$
- **weights**: vector of quadrature weights corresponding to the knots above
- **size**: size of the sparse grid, i.e. the number of non-repeated knots
- **m**: 0-based index array that maps each knot of $\mathbf{Sr}.\text{knots}$ to their original position in $\mathbf{S}.\text{knots}$ (if they have been retained as unique representative of several repeated knots)
- **n**: 0-based index array that maps each knot of $\mathbf{S}.\text{knots}$ to $\mathbf{Sr}.\text{knots}$

In sgpykit, these grids are stored as customizable Structure Arrays, where additional fields can be added by the user.

### Modify the Domain of a Sparse Grid

It is easy to modify the domain of a sparse grid from $(-1,1)^N$ to other hyper-rectangles.

In [ ]:
N=2
w=4
# generate knots on the desired hyper-rectangle (here (0,2)^2 )
knots = lambda n: sg.knots_CC(n,0,2,'nonprob')
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling);

# plot the grid itself
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], '.k', label='grid S')  # or 'DisplayName','grid S'

plt.legend()

One can mix different intervals and different knot families on different directions.

In [ ]:
N=2
knots1 = lambda n: sg.knots_CC(n,0,2,'nonprob')
knots2 = lambda n: sg.knots_uniform(n,-1,5,'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],[sg.lev2knots_doubling, sg.lev2knots_lin])

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, S, [], '.k', label='grid S2')
plt.legend()

## Reduce a Sparse Grid

The tensor grids forming the sparse grid may have points in common (even when using non-nested points).
To save computational time during, e.g., evaluation of a function on a sparse grid, it is important
to get rid of these repetitions. To this end, use the function [`reduce_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.reduce_sparse_grid). The quadrature weights
are of course consistently modified. The field "size" tells the number of points in the reduced grid.

In [ ]:
N=2
w=1
knots1 = lambda n: sg.knots_CC(n,-1,1,'nonprob')


lev2nodes, idxset = sg.define_functions_for_rule('SM',N)
S,_ = sg.create_sparse_grid(N, w, knots, lev2nodes, idxset)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
N=2
w=1
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
a = np.array([[1,2]])
b = a[0,:]
b += 1
a

In [ ]:
S.m

In [ ]:
Sr

In [ ]:
print(f'size of original grid: {np.shape(sg.misc.reshape_nested_lists_to_nrows(S.knots))}')
print(f'size of reduced  grid: {Sr.knots.shape}')
print(f'Sr.size: {Sr.size}')

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
h = sg.plot_sparse_grid(axs[0], S, [], 'color','b','marker','d','MarkerFaceColor','b','DisplayName','original grid')
axs[0].legend()
h = sg.plot_sparse_grid(axs[1], Sr, [], 'color','b','marker','d','MarkerFaceColor','b','DisplayName','reduced grid')
axs[1].legend()
fig.tight_layout()  # auto spacing subplots

The Kit provides a short-hand to create and reduce a "vanilla sparse grid", i.e.
- Clenshaw-Curtis points in $[-1,1]$
- lev2knots_doubling
- multi-index set: $\sum (ii) \leq w$
(cf [`define_functions_for_rule('SM')`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.idxset_functions.html#sgpykit.tools.idxset_functions.define_functions_for_rule) )

In [ ]:
N = 2
w = 3
S,Sr,_ = sg.create_sparse_grid_quick_preset(N,w)

# PART 2: Evaluate a Function on a Sparse Grid

## Basics

The kit comes with the function [`evaluate_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.evaluate_on_sparse_grid), which allows evaluating a function on the points of a sparse grid and provides recycling of previous evaluations.
It works for both scalar-valued and vector-valued functions. Sparse grids passed as input must be reduced.

In [ ]:
fs = lambda x: sum(x) # or np.sum(x,axis=0)
fv = lambda x: 2*x

N=2
w=3
S,_ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
# plain use of evaluate_on_sparse_grid: no recycling
evals_plain_fs,*_ = sg.evaluate_on_sparse_grid(fs, S=None, Sr=Sr)
evals_plain_fv,*_ = sg.evaluate_on_sparse_grid(fv, S=None, Sr=Sr)

In [ ]:
# a direct computation
pts = Sr.size

os = 1 # np.shape(fs(Sr.knots[:,0])[0]  # fs returns a number, so np.shape makes no sense here
ov = np.shape(fv(Sr.knots[:,0]))[0]

In [ ]:
evals_direct_fs = np.zeros((os,pts))
evals_direct_fv = np.zeros((ov,pts))

for i in range(pts):
    evals_direct_fs[:,i] = fs(Sr.knots[:,i])
    evals_direct_fv[:,i] = fv(Sr.knots[:,i])

assert np.allclose(evals_plain_fs, evals_direct_fs), "Results Mismatch"
assert np.allclose(evals_plain_fv, evals_direct_fv), "Results Mismatch"

## Use Recycling Feature

In [ ]:
f = lambda x: sum(x)

N=2
w=3

S,_ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w=4
T,_ = sg.create_sparse_grid(N,w,lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

In [ ]:
evals_non_rec,*_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Tr)
evals_old, *_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)
evals_rec,*_ = sg.evaluate_on_sparse_grid(f,T,
                                          Sr=Tr,
                                          evals_old=evals_old,
                                          S_old=S,
                                          Sr_old=Sr)
assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## Recycle from a "List of Points"

It is also possible to recycle from a list of points. However, the algorithm used to detect
which points are to be evaluated is much slower than the previous case for large $N$.

In [ ]:
f = lambda x: sum(x)

N=20
w=1

S,_ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w=2
T,_ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

evals_non_rec,*_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Tr)
with sg.misc.time_block():
    evals_old,*_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)
    evals_rec,*_ = sg.evaluate_on_sparse_grid(f,T,
                                              Sr=Tr,
                                              evals_old=evals_old,
                                              S_old=S,
                                              Sr_old=Sr)


In [ ]:
# pretend we only know the list of points Sr.knots, to see the difference in performance ...
with sg.misc.time_block():
    evals_old, *_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)
    evals_rec_slow,*_ = sg.evaluate_on_sparse_grid(f,T,
                                              Sr=Tr,
                                              evals_old=evals_old,
                                              S_old=S,
                                              Sr_old=Sr)

assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## Use Recycling Feature for Vector Output

In [ ]:
f = lambda x: 2*x

N=2
w=1

S,_ = sg.create_sparse_grid(N, w, lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

w=2
T,_ = sg.create_sparse_grid(N,w,lambda n: sg.knots_uniform(n,-1,1), sg.lev2knots_lin)
Tr = sg.reduce_sparse_grid(T)

In [ ]:
evals_non_rec,*_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Tr)
evals_old, *_ = sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)
evals_rec,*_ = sg.evaluate_on_sparse_grid(f,T,
                                       Sr=Tr,
                                       evals_old=evals_old,
                                       S_old=S,
                                       Sr_old=Sr)
assert np.allclose(evals_non_rec, evals_rec), "Results Mismatch"

## Evaluate a Function on a Sparse Grid

(TODO: more test cases / examples here?)

# PART 3: Integration on Sparse Grids - Basics

In this part, we show how to use the Kit to perform high-dimensional quadrature. We consider the
following function, for which we know the analytic expression of the integral:

$$
f(\mathbf{x}) = \prod_{i=1}^N \frac{1}{\sqrt{x_i + b}}, \quad \mathbf{x} \in [-1,1]^N
$$

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)

b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

# generate the knots and the SM grid. 'nonprob' means we are integrating w.r.t. the pdf rho(x)=1 and not rho(x)=1/prod(b_i - a_i)
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
# compute integral
mknots = Sr.knots
mweights = np.transpose(Sr.weights)

I = f(mknots, b) @ mweights
assert np.isclose(I,1.883984044753591), "Results mismatch"

In [ ]:
# alternatively use
I2,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr) # Sr must be reduced here

print('difference between values:', I-I2)

In [ ]:
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

Sometimes, we have access to the evaluations of $f$ from earlier code; then we just need to do the linear combination. The package provides
a convenience wrapper for this purpose, instead of typing `f_vals @ np.transpose(Sr.weights)`.

In [ ]:
print('convenience wrapper')

f_vals = f(Sr.knots, b)
I3,_ = sg.quadrature_on_sparse_grid(f_vals, S=None, Sr=Sr)
I2-I3

The convenience wrapper can also handle the case of computing quadrature for multiple functions at the same time. The values of each function
must be stored as rows of a matrix.

In [ ]:
many_f = np.vstack([f_vals, f_vals, f_vals, f_vals, f_vals])

I4,_ = sg.quadrature_on_sparse_grid(many_f,S=None, Sr=Sr)
I4

## Integration on Tensor Grids

In [ ]:
# f = ... see above
b = 3
N = 2
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# let's build a tensor grid with the following choices
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
ii = np.array([6,5])

m = sg.lev2knots_doubling(ii)
T = sg.tensor_grid(N, m, knots)
S = sg.tensor_to_sparse(T)

# with the call above, S.idx is automatically set to the number of points in each dir,
# which implies S.idx = lev2knots(ii). If you want S.idx = ii, use the following call
# S = tensor_to_sparse(T,ii);

Sr = sg.reduce_sparse_grid(S)
sg.is_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Use Other Quadrature Knots
As already seen in the introduction, other quadrature knots are available.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# let's build a tensor grid with the following choices
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob') # <- now uniform
w = 4
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Modify Quadrature Domain
Suppose integrating over $(-1,3)^N$.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(3+b)-2*np.sqrt(-1+b))  # <- 3 instead of 1
I_ex = I_1d**N

In [ ]:
# generate the knots in (-1,3)
knots = lambda n: sg.knots_CC(n,-1,3,'nonprob')
w = 6
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Compute Moments of Random Variables
Here we compute $E[f(\mathbf{x})] = \int_{[-2, 1]\times[0.5, 6]} f(\mathbf{x}) /(3\cdot 5.5) \, d\mathbf{x}$, where $3\cdot 5.5$ is the size of the domain.

In [ ]:
# f = ... see above
b = 3
N = 2
I_ex = 1/3/5.5*(2*np.sqrt(1+b)-2*np.sqrt(-2+b))*(2*np.sqrt(6+b)-2*np.sqrt(0.5+b))

In [ ]:
# the best-practice is to generate knots on (-2,1) and (0.5,6), specifying 'prob' as input to the
# knots-generatic function
knots1=lambda n: sg.knots_CC(n,-2,1,'prob')  # knots1=@(n) knots_CC(n,-2,1); would work as well as 'prob' is the default value
knots2=lambda n: sg.knots_CC(n,0.5,6,'prob') # knots2=@(n) knots_CC(n,0.5,6); would work as well as 'prob' is the default value

In [ ]:
w = 6
S,_ = sg.create_sparse_grid(N, w, [knots1, knots2], sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)

## Recycle Evaluations from Previously Computed Grids

Just as [`evaluate_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.evaluate_on_sparse_grid), [`quadrature_on_sparse_grid()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.main.html#sgpykit.main.quadrature_on_sparse_grid) provides evaluation recycling.

In [ ]:
f = lambda x,b: np.prod(1./np.sqrt(x+b), axis=0)
b = 5
N = 2
# the starting grid
w = 7
knots = lambda n: sg.knots_CC(n,-2,1,'prob')
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)
IS,evals_S = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)

In [ ]:
# the new grid
w = 8
T,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Tr = sg.reduce_sparse_grid(T)
# the recycling call.
IT_rec,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=T, Sr=Tr, evals_old=evals_S, S_old=S, Sr_old=Sr)

np.testing.assert_almost_equal(IT_rec, IS)

In [ ]:
# the non-recycling call
IT,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Tr)

np.testing.assert_almost_equal([IT_rec, IT], 0.228763833671747)

## How to Build More Complex Sparse Grids: Anisotropic Grids

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

In [ ]:
# specify a rule like in Back Nobile Tamellini Tempone, `Stochastic Spectral Galerkin and Collocation...a  numerical comparison''
rates = [1, 2, 2, 2]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2nodes,idxset = sg.define_functions_for_rule('TD', rates)
w = 4
S2,_ = sg.create_sparse_grid(N,w,knots,lev2nodes,idxset)
Sr = sg.reduce_sparse_grid(S2)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)  # matlab result: 1.456974987823489e-04

## How to Build More Complex Sparse Grids: Use multiidx_box_set
As seen in the introduction, specify directly the set of multi-indices involved.
Here, we generate the box set of all multi-indices $\leq [3, 5, 2, 3]$ in lexicographic order.

In [ ]:
# f = ... see above
b = 3
N = 4
I_1d = (2*np.sqrt(1+b)-2*np.sqrt(-1+b))
I_ex = I_1d**N

C,_ = sg.multiidx_box_set([2, 4, 1, 2])  # X is C without [2 4 1 2]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S3,_ = sg.create_sparse_grid_multiidx_set(C, knots, sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S3)
I,_ = sg.quadrature_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)
# compare with exact value
print('quad error:')
np.abs(I-I_ex)  # matlab result: 6.552113139615123e-04

## Convergence Study

See test_sparse_quadrature.m (not ported to Python yet).

# PART 4: Interpolation on a Sparse Grid

## Basics

The sparse grid also provides an interpolant / surrogate model for the original function. The
interpolant can be evaluated at non-grid points.
All the previous topics (changing the domain, building anisotropic grids, etc.) apply immediately to
the interpolation case.

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 4
w = 8

In [ ]:
# generate the knots and the SM grid. 'nonprob' means we are integrating w.r.t. the pdf rho(x)=1 and not rho(x)=1/prod(b_i - a_i)
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

#non_grid_points=rand(N,100) # TODO ?
non_grid_points = np.hstack((0.5 * np.ones((N, 1)), np.zeros((N, 1))))

function_on_grid = f(Sr.knots, b)

f_values = sg.interpolate_on_sparse_grid(S, Sr, function_on_grid, non_grid_points)
# compare with exact value
print('Interpolation error:',
      np.max( np.abs( f_values-f(non_grid_points,b) ) ))

## Interpolation on a Tensor Grid

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 2
# let's build a tensor grid with the following choices
idx = [10, 8]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2knots = sg.lev2knots_lin

In [ ]:
# create the tensor grid, convert it to sparse
T = sg.tensor_grid(N, lev2knots(idx), knots)
S = sg.tensor_to_sparse(T)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
non_grid_points = np.hstack((0.5 * np.ones((N, 1)), np.zeros((N, 1))))

function_on_grid = f(Sr.knots, b)

f_values = sg.interpolate_on_sparse_grid(S, Sr, function_on_grid, non_grid_points)

# compare with exact value
print('Interpolation error:',
      np.max( np.abs( f_values-f(non_grid_points,b) ) ))  # matlab result: 6.411494807290197e-08

## Interpolation in 1D

For 1D interpolation, a dedicated function exists. It's called [`univariate_interpolant()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.polynomials_functions.html#sgpykit.tools.polynomials_functions.univariate_interpolant) and can operate on
vector-valued functions, like here below, where we interpolate a function with two components, i.e., $F: \mathbb{R} \to \mathbb{R}^2$.

In [ ]:
# the two components of f
f1 = lambda x: x**3
f2 = lambda x: np.sin(2*x)

# interpolation points and values
x_interp = np.linspace(-1, 2, 4)
F_interp = np.vstack([f1(x_interp),
                      f2(x_interp)])

# evaluate the interpolant on a much finer grid
x_eval = np.arange(-1, 2.01, 0.01)
F_eval_interp = sg.univariate_interpolant(x_interp,F_interp,x_eval)

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], x_eval, F_eval_interp[0,:], 'DisplayName', 'interpolant')
sg.plot(axs[0], x_interp, F_interp[0,:],'o','DisplayName','interpolation points')
sg.plot(axs[0], x_eval, f1(x_eval),'DisplayName','true fun')
axs[0].legend()
sg.plot(axs[1], x_eval, F_eval_interp[1,:], 'DisplayName', 'interpolant')
sg.plot(axs[1], x_interp, F_interp[1,:],'o','DisplayName','interpolation points')
sg.plot(axs[1], x_eval, f2(x_eval),'DisplayName','true fun')
axs[1].legend()

fig.tight_layout() 

## Interpolation Error on Sparse Grid Points

Since the sparse grid is a linear combination of several tensor grid interpolants, the interpolation
error at a point of the sparse grid is not necessarily zero, unless all tensor interpolants
include that point.

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 4
# a sparse grid withx non-nested points: interpolation error in sparse grid points will be
# non-zero in general
w = 4

knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

non_grid_points=np.zeros((N,1))

function_on_grid,*_=sg.evaluate_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)

f_values = sg.interpolate_on_sparse_grid(S,Sr,function_on_grid,non_grid_points)

print('Interpolation error - non-nested grid:')
np.max( np.abs( f_values-f(non_grid_points,b) ) )  # matlab result: 8.556139706267230e-06

In [ ]:
# the interpolation error will instead be zero if we use nested points and
# consider e.g. [0 0 0 0] which belongs to all of the tensor grids

knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
T,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Tr = sg.reduce_sparse_grid(T)

non_grid_points = np.zeros((N,1))
function_on_grid = f(Tr.knots,b)

f_values = sg.interpolate_on_sparse_grid(T,Tr,function_on_grid,non_grid_points)

print('Interpolation error - nested grid:')
np.max( np.abs( f_values-f(non_grid_points,b) ) )

## Plot Sparse Grids Interpolant
### Case $N=2$

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=[4, 1]
bb=[6, 5]

# the function to be interpolated
f = lambda x: 1./(1+0.5*sum(x**2))
# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, nb_plot_pts=10)
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
# access to plot handles for further editing is available. E.g., this sets dots to black

fig, axs = sg.figure_create(dims=3)
h = sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab
# Check if h is a list of line objects or a single line object
if isinstance(h, list):
    for line in h:
        line.set_markerfacecolor('black')
else:
    h.set_markerfacecolor('black')

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr, [])

In [ ]:
## as expected, the interpolant might be bad if equispaced point are used

f = lambda x: 1./(1+(5*x[0])**2)*1./(1+(5*x[1])**2)
a=-1; b=1;
domain = np.array([[-1, -1], [1, 1]])
N = 2
w = 6
lev2knots = sg.lev2knots_doubling

# a bad choice: equispaced points, we get them by trap-rule. Build a grid with them
knots_bad = lambda n: sg.knots_trap(n,a,b,'nonprob')
S_bad,_ = sg.create_sparse_grid(N,w,knots_bad,lev2knots)
Sr_bad = sg.reduce_sparse_grid(S_bad)
f_values_bad,*_ = sg.evaluate_on_sparse_grid(f,S=None, Sr=Sr_bad)

# a good choice: CC points. Build another grid with them, to compare. Note that the two grids will have the same
# number of points
knots_ok = lambda n: sg.knots_CC(n,a,b,'nonprob')
S_ok,_ = sg.create_sparse_grid(N,w,knots_ok,lev2knots)
Sr_ok = sg.reduce_sparse_grid(S_ok)
f_values_ok,*_ = sg.evaluate_on_sparse_grid(f,S=None, Sr=Sr_ok)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr_bad,[],'o','MarkerSize',4,'LineWidth',2)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S_bad,Sr_bad,domain,f_values_bad,'with_f_values',nb_plot_pts=40)
axs.view_init(elev=29, azim=240)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr_ok,[],'o','MarkerSize',4,'LineWidth',2)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S_ok,Sr_ok,domain,f_values_ok,'with_f_values',nb_plot_pts=40)
axs.view_init(elev=29, azim=240)

### Case $N=3$

In [ ]:
# define sparse grid over [4,6] x [1,5] x [2 3]
N=3
aa=[4, 1, 2]
bb=[6, 5, 3]

# the function to be interpolated
f = lambda x: 1./(1+0.5*sum(x**2))

# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
knots3 = lambda n: sg.knots_CC(n,aa[2],bb[2],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2,knots3],sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values','nb_plot_pts',10,'nb_contourfs',10,'nb_contourf_lines',40)
#axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
# we have two ways of plotting the sparse grid:
sg.plot3_sparse_grid(Sr, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

In [ ]:
# way 2): two-dimensional projections (they will look identical in this case)
fig, axs = sg.figure_create(nrows=1, ncols=3, figsize=(8, 4))
sg.plot_sparse_grid(axs[0], Sr, [1,2])
sg.plot_sparse_grid(axs[1], Sr, [2,3])
sg.plot_sparse_grid(axs[2], Sr, [1,3])
fig.tight_layout()  # auto spacing subplots

### Case $N>3$

In [ ]:
N = 7
aa = -1 * np.ones(N)
bb = np.ones(N)

# The function to be interpolated
#f = lambda x:
def f(x):
    x = np.atleast_2d(x)
    y = 1 / (1 + 0.5 * x[0, :]**2 + 0.25 * x[1, :]**2 + 5 * x[2, :]**2 + 
                  2 * x[3, :]**2 + 0.001 * x[4, :]**2 + 10 * x[5, :]**2 + 10 * x[6, :]**2)
    return y
# f = lambda x: 1 / (1 + 0.5 * x[0, :]**2 + 0.5 * x[1, :]**2 + 0.5 * x[2, :]**2 + 
#                  0.5 * x[3, :]**2 + 0.5 * x[4, :]**2 + 0.5 * x[5, :]**2 + 0.5 * x[6, :]**2)

domain = np.vstack((aa, bb))
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
w=6
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
# add f_values. Note that there are possibly several points which share the values of the coordinates in the cuts,
# therefore there will be points not on the surface. This helps understanding the fluctuations of the function
# when the coordinates not in the cut are not fixed to their average value. In this specific example, changing the
# values of the frozen variables from their averages happens to lower the value of the function. The function generates
# one new figure per cut

fig, axs = sg.figure_create(nrows=1, ncols=3, figsize=(16, 12), dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')

In [ ]:
# specify cuts. Again, because we are specifying cuts, a new figure per cut is generated.  The code below generates two figures
# (the first one is empty)
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(16, 12), dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid,'two_dim_cuts',[1, 4, 2, 7])

In [ ]:
# we have two ways of plotting the sparse grid:
sg.plot3_sparse_grid(Sr, [1,2,4], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

In [ ]:
# way 2): two-dimensional projections
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr, [1,2])

## Convergence Study

See test_sparse_interpolation.m.

# PART 5: Compute the g-PCE of a Function Given its Sparse Grid Approximation

The kit provides a function to compute the generalized Polynomial Chaos Expansion (g-PCE) of a function
of several variables, i.e. the expansion of $f$ in terms of a sum of orthonormal polynomials.

| Supported Random Variable | Orthonormal Polynomials      |
|---------------------------|------------------------------|
| Uniform                   | Legendre, Chebyshev          |
| Normal                    | Hermite                      |
| Exponential               | Laguerre                     |
| Gamma                     | Generalized Laguerre         |
| Beta                      | Jacobi                       |

The coefficients of these expansions are defined as suitable integrals over the space of parameters, and
could thus be approximated with sparse grid quadrature. However, a more efficient technique can be
applied, and it is actually implemented in the Kit. It consists in rearranging the sparse grid
interpolant, which is a linear combination of Lagrange polynomials, as a summation of orthonormal
polynomials (i.e. performing a change of basis to express the same polynomial). Given the relations
between sparse grids and orthogonal expansion, it is always possible to tune the sparse grid so to
obtain the gPCE in a precise polynomial space.

See e.g. Back Nobile Tamellini Tempone, `Stochastic Spectral Galerkin and Collocation...a  numerical
comparison'' for more details on the sparse grid/orthogonal expansion relation and Tamellini ph.D.
thesis, chap.6 or MOX report 13/2012 by Formaggia Guadagnini Imperiali Lever Porta Riva Scotti Tamellini
for details on the conversions.

More examples with different kinds of random variables / orthogonal polynomials can be found in test_convert_to_modal.m.

In [ ]:
# the sparse grid
N = 2
w = 5
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2knots = sg.lev2knots_lin
idxset = lambda i: np.prod(i+1, axis=0)

S,_ = sg.create_sparse_grid(N, w, knots, lev2knots, idxset)
Sr = sg.reduce_sparse_grid(S)

# the domain of the grid
domain = np.vstack((-np.ones(N), np.ones(N)))

# compute a legendre polynomial over the sparse grid
X = Sr.knots
nodal_values = 4*sg.lege_eval_multidim(X,[4, 0],-1,1)+ 2*sg.lege_eval_multidim(X,[1, 1],-1,1)

# conversion from the points to the legendre polynomial. I should recover it exactly
modal_coeffs,K = sg.convert_to_modal(S, Sr, nodal_values, domain, 'legendre')

np.hstack((K, modal_coeffs))

# PART 6: Sparse-Grids-Based Sensitivity Analysis

## Compute Sobol Indices of a Function

In [ ]:
import numpy as np

# Define the domain
aa = np.array([-1, -1, -1])
bb = np.array([1, 1, 1])

# Define the functions
def f1(x):
    x = np.atleast_2d(x)
    return 1 + x[0, :]**2 + x[1, :]**2 + x[2, :]**2

def f2(x):
    x = np.atleast_2d(x)
    return 1 + 5 * x[0, :]**2 + x[1, :]**2 + x[2, :]**2

def f3(x):
    x = np.atleast_2d(x)
    return 1 / (1 + x[0, :]**2 + x[1, :]**2 + x[2, :]**2)

def f4(x):
    x = np.atleast_2d(x)
    return 1 / (1 + 5 * x[0, :]**2 + x[1, :]**2 + x[2, :]**2)

def f(x):
    return np.vstack((f1(x), f2(x), f3(x), f4(x)))

# We expect to see these results:
#   f1: has no mixed effects, so the principal and total Sobol indices are identical. Also, it's isotropic, so the indices of each variable are identical
#   f2: no mixed effects as f1, but y_1 contributes more to the variability of f so it has a larger Sobol total/principal index
#   f3: this function has mixed effects (partial derivatives are nonzero),  so the principal and total Sobol index will be different, but equal among random variables
#   f4: mixed effects, and y_1 contributes more to the variability of f so it has larger Sobol indices

# Generate a sparse grid
domain = np.vstack((aa, bb))
knots = lambda n: sg.knots_CC(n, -1, 1, 'nonprob')
N = len(aa)
w = 5
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_ = sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

# Compute Sobol indices
Sob_i1, Tot_Sob_i1, Mean1, Var1 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[0, :], domain, 'legendre')
Sob_i2, Tot_Sob_i2, Mean2, Var2 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[1, :], domain, 'legendre')
Sob_i3, Tot_Sob_i3, Mean3, Var3 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[2, :], domain, 'legendre')
Sob_i4, Tot_Sob_i4, Mean4, Var4 = sg.compute_sobol_indices_from_sparse_grid(S, Sr, values_on_grid[3, :], domain, 'legendre')

# Display results
print('      f1   |    f2    |   f3    |    f4   ')
print('Principal Sobol indices')
print(np.column_stack((Sob_i1, Sob_i2, Sob_i3, Sob_i4)))
print('Total Sobol indices')
print(np.column_stack((Tot_Sob_i1, Tot_Sob_i2, Tot_Sob_i3, Tot_Sob_i4)))

## Compute Gradients of a Sparse Grid Interpolant (by Finite Differences)

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=np.array([4, 1])
bb=np.array([6, 5])

# the function to be interpolated and its derivatives
def f(x):
    y = 1./(1+0.5*sum(x**2))
    return y
    #return y[0]

def df1(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[0,:]
    return y

def df2(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[1,:]
    return y


# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)


# compute values with function
Grads = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)


# error and visualization

print(max(abs(Grads[0,:] - df1(eval_points))))
print(max(abs(Grads[1,:] - df2(eval_points))))

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], Grads[0,:],'-o','DisplayName','Finite Diff')
sg.plot(axs[0], df1(eval_points),'-','DisplayName','true val')
axs[0].legend()

sg.plot(axs[1], Grads[1,:],'-o','DisplayName','Finite Diff')
sg.plot(axs[1], df2(eval_points),'-','DisplayName','true val')
axs[1].legend()

fig.tight_layout() 

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=np.array([4, 1])
bb=np.array([6, 5])

# the function to be interpolated and its derivatives
def f(x):
    y = 1./(1+0.5*sum(x**2))
    return y
    #return y[0]

def df1(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[0,:]
    return y

def df2(x):
    y = -1./((1+0.5*sum(x**2))**2)*2*0.5*x[1,:]
    return y


# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)


# compute values with function
Grads = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)


# error and visualization

print(max(abs(Grads[0,:] - df1(eval_points))))
print(max(abs(Grads[1,:] - df2(eval_points))))

In [ ]:
# h is computed automatically in each direction as (b-a)/1E5, but can be adjusted if needed.
# In the example below, the length of the interval along direction 1 is O(1E-5) so choosing
# the default h would lead to h = O(1E-10), which incurs in numerical cancellations.
# Thus, setting manually a larger value for h helps in reducing the error

N=2

aa=np.array([4E-5, 1])
bb=np.array([6E-5, 5])

# the function to be interpolated and its derivatives
# ... same as above

# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 5; #
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling);
Sr = sg.reduce_sparse_grid(S)


values_on_grid,*_=sg.evaluate_on_sparse_grid(f,S=None,Sr=Sr)

# generate M random points in the domain where we evaluate the derivative of the sparse grid
# and the true derivative, to check error
M=100
# use get interval map to go from [-1,1]^N to actual domain
my_map=sg.get_interval_map(aa,bb,'uniform')

rng = np.random.default_rng(seed=42)
eval_points = my_map(rng.random(size=(N,M))*2-1)

# compute values with function
Grads_def = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points)
h=np.array([1E-7, 1E-5])
Grads_man = sg.derive_sparse_grid(S,Sr,values_on_grid,domain,eval_points,h)

In [ ]:
# error and visualization
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], Grads_def[0,:],'-o','DisplayName','Finite Diff, default h')
sg.plot(axs[0], Grads_man[0,:],'x','DisplayName','Finite Diff, manual h')
sg.plot(axs[0], df1(eval_points),'-','DisplayName','true val')
axs[0].legend()

sg.plot(axs[1], Grads_def[1,:],'-o','DisplayName','Finite Diff, default h')
sg.plot(axs[1], Grads_man[1,:],'x','DisplayName','Finite Diff, manual h')
sg.plot(axs[1], df2(eval_points),'-','DisplayName','true val')
axs[1].legend()

fig.tight_layout()

In [ ]:
# Error
print(max(abs((Grads_def[0,:] - df1(eval_points))/df1(eval_points))))  # matlab: 1.9397
print(max(abs((Grads_man[0,:] - df1(eval_points))/df1(eval_points))))  # matlab: 0.0046492


## A function to compute Hessians of a function (by finite differences) is also available; see hessian_sparse_grid().

# PART 7: Save Sparse Grid on File

In [ ]:
N = 3

aa = [4, 1, -2]
bb = [6, 5, -1]

knots1 = lambda n: sg.knots_CC(n, aa[0], bb[0], 'nonprob')
knots2 = lambda n: sg.knots_CC(n, aa[1], bb[1], 'nonprob')
knots3 = lambda n: sg.knots_uniform(n, aa[2], bb[2], 'nonprob')

w = 2
S,_ = sg.create_sparse_grid(N, w, [knots1, knots2, knots3], sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

# Save points to 'points.dat'. The first row actually contains two integer
# values, i.e., Sr.size and N
sg.export_sparse_grid_to_file(Sr)

# Save points to 'mygrid.dat'
sg.export_sparse_grid_to_file(Sr, 'mygrid2.dat')

# Save points and weights to 'mygrid_with_weights.dat'
sg.export_sparse_grid_to_file(Sr, 'mygrid_with_weights2.dat', with_weights=True)